# 第4周 · 周日：周复习——数据工程思维的完整构建

> 难度：⭐⭐⭐ | 定位：回顾第4周全部知识，检验数据工程思维体系的完整性
> 前置依赖：本周全部6天课程 | 后续用途：为第5-6周SQL分析实战打下数据基础设施认知

---

## 📘 区块1：本周学习地图——六大模块的关联

本周你学的不只是"数仓知识"——你是在构建一种全新的思维方式：**数据工程思维**。

这种思维方式让你从"我只是写SQL的人"变成"我理解数据背后发生了什么"——这是高级数据分析师和初级SQL写手之间最大的分水岭。

```
本周六大模块的关联图：

Day1: 数据从哪来（链路认知）
  ↓ 知道了数据要"流转"——但流转后怎么组织？
Day2: 数仓分层与星型模型（结构设计）
  ↓ 知道了表要分层——但数据会变怎么办？
Day3: SCD与宽表（变化管理）
  ↓ 知道了维度要管理——但大数据量下怎么存储和查询？
Day4: Hive与大数据处理（计算引擎）
  ↓ 知道了Hive怎么跑——但为什么有时跑不动？
Day5: 数据倾斜与查询优化（性能保障）
  ↓ 知道怎么诊断和解决倾斜——但这些知识在实际中怎么组合？
Day6: 数仓综合练习（知识融合）
  ↓ 把前五天的知识融为一个完整的数仓设计方案
Day7: 周复习（体系构建）← 今天
```

### 核心认知：什么是"数据工程思维"

数据工程思维 = 数据流转意识 + 数据结构意识 + 数据质量意识 + 数据性能意识

- **数据流转意识**（Day1）：知道数据从哪来、经过哪、到哪去——能排查延迟和数据差异
- **数据结构意识**（Day2-3）：知道表应该怎么组织、分层、关联——能设计合理的数据方案
- **数据质量意识**（Day3）：知道数据会变、口径会乱、历史需要保留——能设计可信赖的数据体系
- **数据性能意识**（Day4-5）：知道大数据量下SQL怎么跑、怎么优化——能写"跑得通且跑得快"的SQL

这四种意识合在一起，就是你本周最大的收获。

## 📘 区块2：每日核心知识回顾

### Day 1 · 周一：数据从哪来——从业务动作到数据表

**核心认知**：数据分析师不是"SQL Boy"——你的价值在于"理解数据背后的业务含义和流转过程"。

**必背知识点**：
1. 数据流转完整链路：业务系统(OLTP) → 数据采集(Canal/Flume) → 消息队列(Kafka) → ETL处理 → 数仓存储(ODS→DWD→DWS→ADS) → 数据应用
2. OLTP vs OLAP：OLTP支撑业务（毫秒级读写），OLAP支撑分析（批量扫描）——必须物理分离
3. ETL vs ELT：传统先转换再加载，现代大数据先加载再转换
4. 数据延迟排查：从业务系统→采集→Kafka→ETL→数仓→应用，逐层排查
5. 数据"对不上"时：80%是因为口径不同，先对齐口径再怀疑技术

**关键面试题**：
- "你们公司的数据是怎么从业务库到数仓的？"
- "OLTP和OLAP的区别是什么？为什么不能直接在业务库跑分析SQL？"
- "数据延迟了你怎么排查？"

### Day 2 · 周二：数仓建模一——四层分层与维度建模

**核心认知**：分层的本质是把"清洗"和"计算"在数据入库时就做好，让所有人拿到的是干净且口径统一的数据。

**必背知识点**：
1. 四层分层：ODS(照搬原始)→DWD(清洗标准化)→DWS(轻度汇总)→ADS(应用定制)
2. ODS不能跳过——它是可追溯性和断点恢复的基础
3. 事实表vs维度表：事实表记录事件(亿级行、追加)，维度表记录实体属性(万级行、更新)
4. 星型模型=事实表在中间+维度表围绕——查询高效、结构清晰
5. DWD vs DWS选择：维度在DWS中→用DWS(快100倍)；不在→用DWD(灵活但慢)；常用但不在→推动加

**关键面试题**：
- "数仓为什么分四层？每层作用是什么？"
- "事实表和维度表有什么区别？"
- "你怎么决定一个指标该放在DWD还是DWS？"

### Day 3 · 周三：数仓建模二——SCD与宽表设计

**核心认知**："数据会变"是常态。好的数仓设计让你能回答"这个用户上个月的状态是什么"——因为变化被保留了下来。

**必背知识点**：
1. SCD = Slowly Changing Dimension——维度属性会随时间变化
2. Type 1(覆盖)：不需要追溯→纠错型修改
3. Type 2(新增行+时间区间)：完整历史→主力方案，JOIN时加日期范围条件
4. Type 3(新增列)：只看前一个值→偶尔变化
5. 宽表要"刚好够用"：DWD用窄表(规范)、ADS做宽表(方便用)
6. 数据回刷需要通知机制——维护数据信任

**关键面试题**：
- "什么是SCD？Type 1/2/3的区别和适用场景？"
- "用户等级变化用什么SCD？Type 2的JOIN怎么写？"
- "如果历史数据因为回刷变了，你怎么跟业务方解释？"

### Day 4 · 周四：Hive与大数据处理

**核心认知**：Hive不是"慢的MySQL"——它是为PB级离线分析设计的分布式SQL引擎。在大数据环境下，"语法正确"不等于"执行正确"。

**必背知识点**：
1. Hive = SQL翻译成MapReduce/Spark任务→多机并行→汇总
2. 分区=按字段分目录存数据→分区裁剪减少99%的扫描量
3. LIKE不会触发分区裁剪！用=或>=替代
4. 列式存储(ORC/Parquet)→只读需要的列→SELECT * 是浪费
5. MAPJOIN=小表加载到内存→消除Shuffle→JOIN大表小表的终极优化
6. Hive vs Spark vs Presto：ETL用Hive，交互用Presto，ETL+交互用Spark

**关键面试题**：
- "Hive和MySQL的核心区别是什么？"
- "分区裁剪是什么？为什么LIKE不触发？"
- "MAPJOIN的原理和适用条件？"

### Day 5 · 周五：数据倾斜与查询优化

**核心认知**：一个SQL跑不动，80%是因为数据倾斜。加机器解决不了倾斜——9个节点在发呆，加100个也是101个在发呆。

**必背知识点**：
1. 数据倾斜=某个key数据量特别大→单节点拖慢全体
2. 三种倾斜场景：GROUP BY倾斜、JOIN倾斜、COUNT(DISTINCT)倾斜
3. 四种处理方法：过滤(NULL不需要时)、拆分(单独处理倾斜key)、加盐(给key加随机后缀打散)、改写(COUNT DISTINCT→GROUP BY+COUNT)
4. 先找到倾斜的key → 再选处理策略 → 不要无脑加资源
5. 大促前预判：数据量增长+key分布变化→提前重写SQL

**关键面试题**：
- "什么是数据倾斜？怎么发现？"
- "GROUP BY倾斜怎么处理？"
- "COUNT(DISTINCT)为什么慢？怎么优化？"

### Day 6 · 周六：数仓综合练习

**核心认知**：把本周全部知识融合为一个完整的数仓设计方案——覆盖分层、SCD、分区、优化、防倾斜。

**必背知识点**：
1. 完整方案：ODS(同步原始)→DWD(清洗+标准化+SCD)→DWS(按需汇总)→ADS(看板+报表+宽表)
2. 推动建设的逻辑：量化损失→提出MVP→计算ROI→争取支持
3. 数仓边界：不适合实时查询、不适合在线业务、不适合小数据

## 📘 区块3：知识体系自检——30秒内说出正确的应对方案

对于以下每个真实场景，你能在30秒内给出正确判断和分析方向吗？

| # | 场景 | 涉及的Day | 正确的应对方向 |
|---|------|----------|---------------|
| 1 | "数据延迟了，昨天DAU还没出来" | Day1 | 逐层排查：ADS→DWS→DWD→ODS→Kafka→采集→业务系统 |
| 2 | "两个数据源的GMV差3%" | Day1 | 先确认口径一致→再排查技术问题 |
| 3 | "每天跑同一个分析SQL要30秒" | Day2 | 检查这个分析能不能查DWS层→推动汇总到DWS |
| 4 | "去年同期的数据这周变了" | Day3 | 是不是数据回刷了？回刷有没有通知机制？ |
| 5 | "用Hive查1天的数据也很慢" | Day4 | 分区裁剪是否触发？有没有用LIKE？ |
| 6 | "同样的SQL昨天跑2分钟今天2小时" | Day5 | 数据量涨了多少？key分布有没有倾斜？大促/活动？ |
| 7 | "需要设计一个新业务的数据方案" | Day2-3,6 | 分层→事实表/维度表→SCD→分区→汇总→应用 |
| 8 | "这个指标能不能做到实时？" | Day1,4 | 看需求：监控需要实时(T+0)，日报T+1足够 |
| 9 | "团队不知道数仓有什么表" | Day2 | 写数据字典、整理常用表清单、推动命名规范 |
| 10 | "Hive跑SQL太慢了，想换ClickHouse" | Day4 | 看场景：交互式→ClickHouse正确，大规模ETL→Hive更合适 |

如果这10个场景你都能快速判断——恭喜，你已经具备了数据工程思维。
如果还有迟疑——回到对应天的内容重新学习。

## 📘 区块4：阶段回顾——前四周全部知识的全景图

第4周完结，你完成了学习计划的两个阶段：

```
┌─────────────────────────────────────────────────────┐
│              阶段一：商业分析思维（W1-W3）              │
│                                                      │
│  W1 指标与度量                                        │
│    五步循环 / 好指标vs虚荣指标 / OSM / CMOP           │
│    指标拆解 / 异动分析                                 │
│                                                      │
│  W2 增长分析框架                                      │
│    AARRR / 同期群 / 漏斗分析 / 留存与LTV              │
│    RFM分群 / PDS日活拆解                              │
│                                                      │
│  W3 系统化分析                                        │
│    归因模型 / ISS框架 / 补贴ROI                       │
│    数据产品 / 冷启动策略 / 数据质量                    │
│                                                      │
│  → 你学会了：用什么框架分析什么问题                     │
├─────────────────────────────────────────────────────┤
│              阶段二：数据工程思维（W4）                 │
│                                                      │
│  W4 数仓思维                                          │
│    数据流转(OLTP→OLAP) / 分层(ODS/DWD/DWS/ADS)       │
│    SCD(缓慢变化维度) / 宽表设计 / Hive大数据           │
│    数据倾斜 / 查询优化 / 数仓方案设计                   │
│                                                      │
│  → 你学会了：分析所用的数据从哪来、怎么存、怎么优化       │
├─────────────────────────────────────────────────────┤
│              阶段三：SQL分析实战（W5-W6 即将开始）       │
│                                                      │
│  → 用前两个阶段的知识，在真实数据上解决业务问题           │
│  → 窗口函数、连续登录、JOIN技巧、Cohort分析             │
│  → TopN问题、留存计算、漏斗与转化                       │
└─────────────────────────────────────────────────────┘
```

### 两个阶段的关系

**阶段一（W1-W3）教你"想清楚分析什么"**——商业框架给你分析的方向：
- GMV跌了 → 异动分析 + 指标拆解
- 用户流失了 → AARRR + 同期群
- 要选哪个渠道投放 → 归因模型

**阶段二（W4）教你"搞明白数据从哪来"**——数据工程思维告诉你分析的"原材料"：
- 这个数据是实时的还是T+1的？
- 这个表在数仓的哪一层？数据质量如何？
- 这个SQL能跑动吗？会不会倾斜？

**两者结合 = 能"想清楚"也能"做出来"的高阶分析师。**

缺少阶段一 → 你是"SQL写得很快但不知道为什么写"的取数工具人
缺少阶段二 → 你是"分析思路很好但数据一到手就卡住"的纸上谈兵型分析师
两者兼备 → 你能从"业务问题"追溯到"数据方案"再到"SQL实现"再到"分析结论"——完整的价值闭环

## 📘 区块5：下周预告

### 第5-6周：SQL分析思维——在真实数据上解决业务问题

接下来两周，你将：
- **W5**：窗口函数 / 连续登录 / JOIN全家桶 / CTE / 日期与时间序列 / 字符串与正则 / SQL性能优化
- **W6**：复杂业务问题的SQL解法 / TopN / 留存计算 / 漏斗SQL / Cohort分析

但是注意——你写这些SQL时，已经带着本周的"数据工程思维"了：
- 写SQL前你会先想：这个分析查DWS还是DWD？
- 看到查询慢你会先检查：分区裁剪触发了吗？有没有倾斜？
- 数据对不上你会先怀疑：口径一致吗？用的是Type 1还是Type 2？

这就是第4周的意义——它不是让你放下前3周的商业思维，而是在商业思维上叠加一层"数据工程思维"。两种思维叠加，你就不再是"只会写SQL的人"——你是"理解数据、理解业务、能写高效SQL"的数据分析师。

## 📘 区块6：本周核心能力对照

| 能力 | Day | 初级（能自己做） | 中级（能解释给同事） | 高级（能设计推动） |
|------|-----|-----------------|---------------------|-------------------|
| 数据流转认知 | D1 | 画出流转链路 | 排查延迟根因 | 推动时效性优化 |
| 分层建模能力 | D2 | 判断表的层级 | 设计部门级分层 | 推动高频查询进DWS |
| SCD设计能力 | D3 | 区分Type 1/2/3 | 选择合适的SCD | 修复不当的SCD设计 |
| Hive优化能力 | D4 | 写出分区裁剪SQL | 使用MAPJOIN优化 | 设计完整分区策略 |
| 倾斜处理能力 | D5 | 识别倾斜风险 | 用加盐/拆分处理 | 大促前预判和预防 |
| 方案设计能力 | D6 | 理解已有数仓 | 设计简单方案 | 推动数仓建设 |

### 如果某个能力还停留在"初级"——回顾对应天的区块2（知识点拆解）和区块3（动手练习）。

### 如果某个能力达到了"高级"——可以尝试在工作中运用：推动一个汇总进入DWS层、修复一个SCD问题、优化一条慢SQL。

## 📝 区块7：当日小结

> **第4周为你构建了"数据工程思维"的四根支柱：数据流转意识（知道数据从哪来）、数据结构意识（知道表该怎么设计）、数据质量意识（知道数据会变且变化需要被管理）、数据性能意识（知道大数据下的SQL怎么优化）。这四种意识加上前3周构建的商业分析框架，让你成为"既懂业务又懂数据还懂工程"的三栖数据分析师。下周开始，你将带着这套完整的思维体系进入SQL分析实战——不再只是"写SQL"，而是用SQL去"解决真实的业务问题"。**

**本周三条金句汇总**：
1. (Day1) 数据分析师的价值不在于"SQL写得多快"，而在于"理解数据背后的业务含义和流转过程"
2. (Day2) 分析师推动一个汇总进入DWS层，等于给全团队每天省下N个30秒
3. (Day3) 如果你用"现在的用户等级"去分析"去年的订单"——你看到的不是去年的真相，而是今年的幻觉
4. (Day4) 分区裁剪是Hive优化的第一性原理——一个字符的LIKE和BETWEEN之差，查询时间差100倍
5. (Day5) 加机器解决不了数据倾斜——9个人在发呆，加100个人也是101个人在发呆
6. (Day6) 推动数仓建设不讲技术讲ROI——"每天省4小时"比"我们需要Hive+ORC+分区"更有说服力

### 🔖 课后拓展

1. **总结**：用你自己的话写一份"数仓知识速查卡片"——包含分层命名规范、SCD选择框架、分区裁剪写法、倾斜处理方法

2. **实践**：找到你工作中一条慢SQL，运用本周知识从分区裁剪、JOIN方式、倾斜检测三个角度分析并优化

3. **预习**：下周一是SQL窗口函数——注意回顾W1你学的基础知识，窗口函数在W5会深度展开

---

*03analyst_thinking · Week 4 Day 7 · 周复习*

> 扩展阅读：
> - 想深入数仓技术细节 → 02analyst_thinking W2D1-6（Hive/数仓专题）
> - 想回顾前3周商业框架 → 02analyst_thinking W1(指标体系)/W3(增长分析)
> - 想同步加强Python技能 → 01python_learn W4（缺失值/异常值/特征工程）